# Data Ingestion & Document Loading

This notebook introduces the **data ingestion** foundation of the `TraditionalRAG` project using LangChain. Before text can be split, embedded, and stored in a vector database (covered in `pdf_loader.ipynb`), raw data must be loaded into LangChain's standardized `Document` representation.

## Notebook Workflow
1. **Document Structure** — Understand and manually construct a `langchain_core.documents.Document`.
2. **Sample Data Generation** — Create local `.txt` files under `../data/text_files/`.
3. **Single-File Loading (`TextLoader`)** — Ingest an individual plain-text file.
4. **Directory Loading (`DirectoryLoader` + `TextLoader`)** — Batch-load all `.txt` files from a folder.
5. **PDF Loading (`DirectoryLoader` + `PyMuPDFLoader`)** — Batch-load multi-page `.pdf` files from `../data/pdf/`.

## 1. LangChain `Document` Structure

A LangChain `Document` is the core data container passed through RAG pipelines. Every `Document` contains two fields:
- **`page_content` (`str`)**: The main textual content that will later be chunked and embedded.
- **`metadata` (`dict`)**: Contextual attributes (such as `source`, `pages`, `author`, or `date_created`) used for provenance, citations, and metadata-based filtering during retrieval.

In [2]:
# Document Structure
# Import LangChain's standard Document schema for wrapping text content and metadata

from langchain_core.documents import Document

In [5]:
# Instantiate a sample Document manually with text content and custom metadata
doc = Document(
    page_content = "This is the main text content which I am using to create a RAG",
    metadata = {
        "source": "example.txt",
        "pages": 1,
        "author": "Raunak Manna",
        "date_created": "2026-11-17",
    }
)
# Display the constructed Document representation
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Raunak Manna', 'date_created': '2026-11-17'}, page_content='This is the main text content which I am using to create a RAG')

## 2. Creating Sample Text Files

To demonstrate file-based and directory-based ingestion, we first ensure the target directory `../data/text_files` exists and write two UTF-8 encoded sample `.txt` files (`python_intro.txt` and `machine_learning.txt`).

In [13]:
## create a simple txt file

import os
# Create the target directory for sample text files if it does not already exist
os.makedirs("../data/text_files", exist_ok=True)

In [15]:
## create sample text files

# Mapping of destination file paths to their sample plain-text content
sample_texts = {
    "../data/text_files/python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",

    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


    """

}

# Write each sample document to disk using UTF-8 encoding
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


## 3. Single-File Ingestion with `TextLoader`

`TextLoader` reads a single plain-text file from disk and wraps its full text into a list containing one `Document` object. The `source` file path is automatically populated inside `metadata`.

In [25]:
### Text Loader
from langchain_community.document_loaders import TextLoader

# Configure TextLoader for a single file with explicit UTF-8 encoding
loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
# Load the file into a list of LangChain Document objects
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


## 4. Batch Text Ingestion with `DirectoryLoader`

When ingesting multiple files, `DirectoryLoader` recursively scans a directory using a `glob` pattern and applies a specified loader class (`loader_cls`) to each matched file.
- `glob="**/*.txt"` matches all `.txt` files in the directory and any subdirectories.
- `loader_cls=TextLoader` delegates parsing of each matched file to `TextLoader`.
- `loader_kwargs={"encoding": "utf-8"}` forwards keyword arguments to each `TextLoader` instance.

In [ ]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

# Recursively discover and load all .txt files from the sample text directory
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

# Execute batch loading and display the resulting list of Document objects
documents = dir_loader.load()
documents

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervise

## 5. Batch PDF Ingestion with `DirectoryLoader` & `PyMuPDFLoader`

`PyMuPDFLoader` (powered by PyMuPDF / `fitz`) extracts text and rich document metadata from PDF files. Unlike `TextLoader` (which produces one `Document` per file), `PyMuPDFLoader` outputs **one `Document` per page**, automatically attaching PDF metadata such as `page`, `total_pages`, `title`, `source`, `file_path`, and timestamps.

In [26]:
### PDF Loader
from langchain_community.document_loaders import PyMuPDFLoader

# Recursively discover all PDF files under ../data/pdf and parse each page via PyMuPDFLoader
dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
)

# Load all PDF pages into a flat list of LangChain Document objects
documents = dir_loader.load()
documents

[Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-11-10T03:23:09+00:00', 'source': '../data/pdf/Debkanta Study Zone - TCS NQT 2027 Complete Placement Guide (1).pdf', 'file_path': '../data/pdf/Debkanta Study Zone - TCS NQT 2027 Complete Placement Guide (1).pdf', 'total_pages': 12, 'format': 'PDF 1.4', 'title': 'Debkanta Study Zone — TCS NQT 2027 Complete Placement Guide', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-11-10T03:23:09+00:00', 'trapped': '', 'modDate': "D:20261110032309+00'00'", 'creationDate': "D:20261110032309+00'00'", 'page': 0}, page_content='2027 BATCH • OFF-CAMPUS & NQT PLACEMENT EDITION\nTCS NQT 2027 Complete Placement\nGuide\nExam Pattern, Roles & Packages, Eligibility Rules, Complete Topic-Wise Syllabus\n(Foundation & Advanced), and the 90-Day First-Attempt Roadmap — all in one\nstructured handbook.\nBy Debkanta Study Zone — TCS Placement Guide\n190 Min\nTOTAL EXAM TIME\n2 Parts\nPART A & PART B\n0\nNEGATI